## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

plt.rcParams.update({"figure.dpi": 120, "font.size": 11})
sns.set_style("whitegrid")

os.makedirs("../models",  exist_ok=True)
os.makedirs("../figures", exist_ok=True)
os.makedirs("../results", exist_ok=True)

RANDOM_SEED = 42
print("Libraries loaded.")

## 2. Load & Inspect Dataset

In [ ]:
df = pd.read_csv("../data/processed/nlrp3_ml_ready_no_leakage.csv")

print(f"Dataset shape : {df.shape}")
print(f"Columns       : {list(df.columns)}")
df.describe().round(3)

In [ ]:
# Ensure all columns are numeric
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna()
print(f"Clean dataset : {df.shape[0]:,} molecules")

## 3. Feature / Target Split

In [ ]:
FEATURE_COLS = ["Molecular Weight", "AlogP", "#RO5 Violations"]
TARGET_COL   = "pIC50"

X = df[FEATURE_COLS].values
y = df[TARGET_COL].values

print(f"Feature matrix X : {X.shape}")
print(f"Target vector  y : {y.shape}")
print(f"pIC50 stats      : mean={y.mean():.2f}, std={y.std():.2f}, min={y.min():.2f}, max={y.max():.2f}")

## 4. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_SEED
)

print(f"Training set  : {X_train.shape[0]:,} molecules")
print(f"Test set      : {X_test.shape[0]:,} molecules")

## 5. Model Definitions

In [ ]:
models = {
    "Random Forest": Pipeline([
        ("scaler", StandardScaler()),
        ("model",  RandomForestRegressor(
            n_estimators=600,
            min_samples_split=5,
            min_samples_leaf=2,
            random_state=RANDOM_SEED,
            n_jobs=-1
        ))
    ]),
    "Gradient Boosting": Pipeline([
        ("scaler", StandardScaler()),
        ("model",  GradientBoostingRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=4,
            random_state=RANDOM_SEED
        ))
    ]),
    "SVR": Pipeline([
        ("scaler", StandardScaler()),
        ("model",  SVR(kernel="rbf", C=10, epsilon=0.1))
    ]),
    "Ridge (baseline)": Pipeline([
        ("scaler", StandardScaler()),
        ("model",  Ridge(alpha=1.0))
    ])
}

print(f"Models to evaluate: {list(models.keys())}")

## 6. Train & Evaluate All Models

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

results = []

for name, pipe in models.items():
    # 5-Fold CV
    cv_scores = cross_val_score(pipe, X_train, y_train,
                                cv=kf, scoring="r2", n_jobs=-1)
    # Train on full train set
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    r2   = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae  = mean_absolute_error(y_test, y_pred)

    results.append({
        "Model"      : name,
        "CV R² (mean)": cv_scores.mean().round(4),
        "CV R² (std)" : cv_scores.std().round(4),
        "Test R²"    : round(r2,   4),
        "Test RMSE"  : round(rmse, 4),
        "Test MAE"   : round(mae,  4),
    })
    print(f"[{name:22s}]  CV R²={cv_scores.mean():.3f}±{cv_scores.std():.3f}  "
          f"Test R²={r2:.3f}  RMSE={rmse:.3f}")

results_df = pd.DataFrame(results).sort_values("Test R²", ascending=False)
results_df

## 7. Best Model: Random Forest – Detailed Analysis

In [ ]:
# Use the Random Forest as primary model
best_model = models["Random Forest"]
y_pred_rf  = best_model.predict(X_test)

r2   = r2_score(y_test, y_pred_rf)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf))
mae  = mean_absolute_error(y_test, y_pred_rf)

print(f"Random Forest – Test Set Performance")
print(f"  R²   : {r2:.4f}")
print(f"  RMSE : {rmse:.4f} pIC50 units")
print(f"  MAE  : {mae:.4f} pIC50 units")

In [ ]:
# Predicted vs Actual scatter plot
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, y_pred_rf, alpha=0.5, color="steelblue", edgecolors="white", s=40)
mn, mx = y_test.min(), y_test.max()
ax.plot([mn, mx], [mn, mx], "r--", linewidth=1.5, label="Ideal")
ax.set_xlabel("Actual pIC50")
ax.set_ylabel("Predicted pIC50")
ax.set_title(f"Random Forest: Predicted vs Actual\n(R² = {r2:.3f}, RMSE = {rmse:.3f})")
ax.legend()
plt.tight_layout()
plt.savefig("../figures/04_rf_predicted_vs_actual.png", bbox_inches="tight")
plt.show()

In [ ]:
# Residual plot
residuals = y_test - y_pred_rf
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(y_pred_rf, residuals, alpha=0.5, color="coral", edgecolors="white", s=40)
ax.axhline(0, color="black", linestyle="--", linewidth=1)
ax.set_xlabel("Predicted pIC50")
ax.set_ylabel("Residual")
ax.set_title("Residual Plot – Random Forest")
plt.tight_layout()
plt.savefig("../figures/05_rf_residuals.png", bbox_inches="tight")
plt.show()

## 8. Feature Importance

In [ ]:
rf_estimator = best_model.named_steps["model"]
importances  = rf_estimator.feature_importances_

fi_df = pd.DataFrame({
    "Feature"   : FEATURE_COLS,
    "Importance": importances
}).sort_values("Importance", ascending=True)

fig, ax = plt.subplots(figsize=(6, 3))
ax.barh(fi_df["Feature"], fi_df["Importance"], color="steelblue")
ax.set_xlabel("Importance Score")
ax.set_title("Feature Importances – Random Forest")
plt.tight_layout()
plt.savefig("../figures/06_feature_importance.png", bbox_inches="tight")
plt.show()

print(fi_df.to_string(index=False))

## 9. Cross-Validation Learning Curve

In [ ]:
from sklearn.model_selection import learning_curve

train_sizes, train_scores, val_scores = learning_curve(
    best_model, X, y,
    cv=5,
    scoring="r2",
    train_sizes=np.linspace(0.1, 1.0, 10),
    n_jobs=-1,
    random_state=RANDOM_SEED
)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(train_sizes, train_scores.mean(axis=1), "o-", color="blue",  label="Training R²")
ax.plot(train_sizes, val_scores.mean(axis=1),   "o-", color="orange",label="Validation R²")
ax.fill_between(train_sizes,
                train_scores.mean(1) - train_scores.std(1),
                train_scores.mean(1) + train_scores.std(1), alpha=0.1, color="blue")
ax.fill_between(train_sizes,
                val_scores.mean(1) - val_scores.std(1),
                val_scores.mean(1) + val_scores.std(1), alpha=0.1, color="orange")
ax.set_xlabel("Training Set Size")
ax.set_ylabel("R² Score")
ax.set_title("Learning Curve – Random Forest")
ax.legend()
plt.tight_layout()
plt.savefig("../figures/07_learning_curve.png", bbox_inches="tight")
plt.show()

## 10. Model Comparison Chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# R² comparison
axes[0].barh(results_df["Model"], results_df["Test R²"], color="steelblue")
axes[0].set_xlabel("Test R²")
axes[0].set_title("Model Comparison – R²")
axes[0].axvline(0, color="black", linewidth=0.5)

# RMSE comparison
axes[1].barh(results_df["Model"], results_df["Test RMSE"], color="coral")
axes[1].set_xlabel("Test RMSE (pIC50 units)")
axes[1].set_title("Model Comparison – RMSE")

plt.tight_layout()
plt.savefig("../figures/08_model_comparison.png", bbox_inches="tight")
plt.show()

## 11. Save Best Model & Results

In [ ]:
# Save the trained Random Forest pipeline
joblib.dump(best_model, "../models/nlrp3_rf_model.pkl")
print("Model saved → models/nlrp3_rf_model.pkl")

# Save model comparison results
results_df.to_csv("../results/model_comparison.csv", index=False)
print("Results saved → results/model_comparison.csv")

# Save feature metadata for use in FDA screening
import json
feature_meta = {
    "feature_cols" : FEATURE_COLS,
    "target_col"   : TARGET_COL,
    "train_size"   : int(X_train.shape[0]),
    "test_size"    : int(X_test.shape[0]),
    "test_r2"      : round(r2,   4),
    "test_rmse"    : round(rmse, 4),
    "test_mae"     : round(mae,  4),
}
with open("../models/model_metadata.json", "w") as f:
    json.dump(feature_meta, f, indent=2)
print("Metadata saved → models/model_metadata.json")

## 12. Summary

In [ ]:
print("=" * 60)
print("  ML MODEL TRAINING SUMMARY")
print("=" * 60)
print(f"  Task          : pIC50 Regression (NLRP3 inhibition)")
print(f"  Train size    : {X_train.shape[0]:,} molecules")
print(f"  Test  size    : {X_test.shape[0]:,} molecules")
print(f"  Features      : {FEATURE_COLS}")
print()
print("  Best Model: Random Forest Regressor")
print(f"    Test R²   : {r2:.4f}")
print(f"    Test RMSE : {rmse:.4f} pIC50 units")
print(f"    Test MAE  : {mae:.4f} pIC50 units")
print("=" * 60)
print()
print(results_df.to_string(index=False))